In [1]:
# This script is modified to use the IGN/CDDIS public FTP mirror for navigation data.

import os
from datetime import datetime
import numpy as np
import requests
import gzip
from glob import glob
import sys
from math import radians, cos, sin, asin, sqrt

# Input parameters
datadir = r'C:\Users\gauth\Documents\SC4000_proj\test' 
stas = ['slac', 'vdcy', 'p222']
obs_url_base = 'https://geodesy.noaa.gov/corsdata/rinex'
nav_url_base = 'https://igs.bkg.bund.de/root_ftp/IGS/BRDC' 
crx2rnx_bin = r"C:\Users\gauth\Documents\RTKLIB\crx2rnx.exe"

# Base station coordinates (lat, lon)
STATIONS = {
    "slac": (37.417, -122.204),
    "vdcy": (34.179, -118.220),
    "p222": (37.539, -122.083),
}

# Known dataset locations
LOCATIONS = {
    "MTV": (37.4046, -122.0764),
    "SVL": (37.3688, -121.9269),
    "SJC": (37.3382, -121.8863),
    "LAX": (33.9416, -118.4085),
    "SFO": (37.6213, -122.3790),
    "OAK": (37.7214, -122.2208),
}

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two lat/lon points"""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    km = 6371 * c
    return km

def get_location_from_dataset(dataset_name):
    """Extract location code from dataset folder name"""
    parts = dataset_name.split('-')
    if len(parts) >= 5:
        return parts[4]  # e.g., "2020-05-15-US-MTV-1" -> "MTV"
    return None

def get_closest_stations(location_code, available_stations):
    """Return list of stations sorted by distance to location"""
    if location_code not in LOCATIONS:
        print(f"Warning: Unknown location {location_code}, using default order")
        return available_stations
    
    loc_lat, loc_lon = LOCATIONS[location_code]
    
    # Calculate distances to all available stations
    distances = []
    for sta in available_stations:
        if sta in STATIONS:
            sta_lat, sta_lon = STATIONS[sta]
            dist = haversine_distance(loc_lat, loc_lon, sta_lat, sta_lon)
            distances.append((dist, sta))
        else:
            distances.append((999, sta))  # Unknown station, put at end
    
    # Sort by distance
    distances.sort()
    sorted_stations = [sta for dist, sta in distances]
    
    print(f"Station order for {location_code}: {[f'{sta}({dist:.1f}km)' for dist, sta in distances[:3]]}")
    return sorted_stations

os.chdir(datadir)

for dataset in np.sort(os.listdir()):
    if not os.path.isdir(dataset):
        continue

    print("Processing:", dataset)
    ymd = dataset.split('-')
    
    # Check for valid date parts before processing
    if len(ymd) < 3:
        print(f"Skipping {dataset}: Invalid date format.")
        continue
        
    try:
        year = ymd[0]
        month = ymd[1]
        day = ymd[2]
        
        # Calculate DOY and get short year
        dt_obj = datetime(int(year), int(month), int(day))
        doy = dt_obj.timetuple().tm_yday
        doy_str = str(doy).zfill(3)
        year_short = year[2:4]
    except ValueError:
        print(f"Skipping {dataset}: Invalid date value.")
        continue
    
    # Get location and sort stations by distance
    location = get_location_from_dataset(dataset)
    sorted_stations = get_closest_stations(location, stas)
    
    # === BEGIN: BASE STATION OBSERVATION DATA BLOCK ===
    # Delete old observation files to force re-download with new closest station
    old_obs_files = glob(os.path.join(dataset,'*.*o')) + glob(os.path.join(dataset,'*.*d'))
    for old_file in old_obs_files:
        try:
            os.remove(old_file)
            print(f"Deleted old file: {os.path.basename(old_file)}")
        except Exception as e:
            print(f"Could not delete {old_file}: {e}")
    
    success = False
    
    # Try stations in order of distance (closest first)
    for idx, station_id in enumerate(sorted_stations):
        # Filename example: slac1360.20d.gz 
        fname_base = station_id + doy_str + '0.' + year_short + 'd.gz'
        
        # Path structure: /YYYY/DOY/station/filename
        obs_url = '/'.join([obs_url_base, year, doy_str, station_id, fname_base])
        
        # *** DEBUG LINE ***
        print(f"Attempting BASE OBS from {station_id} (attempt {idx+1}/{len(sorted_stations)}): {obs_url}")
        # ******************
        
        try:
            # Check if file exists and download/decompress
            response = requests.get(obs_url, timeout=10)
            response.raise_for_status() # Raise exception for 4xx or 5xx status codes
            
            obs = gzip.decompress(response.content) # get obs and decompress
            
            # Decompressed filename: slac1360.20d
            final_obs_filename = fname_base[:-3]
            
            # write obs data
            open(os.path.join(dataset, final_obs_filename), "wb").write(obs)
            print(f"✓ Success: Base OBS file written to {final_obs_filename} using {station_id}")
            success = True
            break  # Exit loop on success

        except Exception as e:
            print(f"✗ Failed {station_id}: {e}")
            continue  # Try next station
    
    if not success:
        print(f"✗✗ FAILED ALL STATIONS for {dataset}")

    # convert crx to rnx (This section runs regardless of failure above)
    crx_files = glob(os.path.join(dataset,'*.*d'))
    if len(crx_files) > 0:
        # Note: This external command execution may fail silently if the executable path is wrong
        # or if the file conversion fails.
        os.system(crx2rnx_bin + ' ' + crx_files[0])
    # === END: BASE STATION OBSERVATION DATA BLOCK ===
    
    
    # === BEGIN: NAVIGATION DATA BLOCK ===
    # Delete old navigation files to force re-download
    old_nav_files = glob(os.path.join(dataset,'*.rnx'))
    for old_file in old_nav_files:
        try:
            os.remove(old_file)
            print(f"Deleted old nav file: {os.path.basename(old_file)}")
        except Exception as e:
            print(f"Could not delete {old_file}: {e}")
    
    # Filename format: BRDC00WRD_R_YYYYDOY0000_01D_MN.rnx.gz
    fname = f'BRDC00WRD_R_{year}{doy_str}0000_01D_MN.rnx.gz'
    
    # Path structure: /daily/YYYY/DOY/filename
    url = '/'.join([nav_url_base, year, doy_str, fname])
    
    # *** DEBUG LINE ***
    print(f"Attempting NAV file download from URL: {url}")
    # ******************
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() 

        obs = gzip.decompress(response.content) 
        
        final_filename = fname[:-3]
        
        open(os.path.join(dataset, final_filename), "wb").write(obs)
        print(f"✓ NAV file downloaded")
        
    except requests.exceptions.RequestException as e:
        print(f'✗ Fail nav (Request Error): {dataset} - {e}')
        
    except Exception as e:
        print(f'✗ Fail nav (General Error): {dataset} - {e}')

Processing: 2021-04-28-US-MTV-2
Station order for MTV: ['slac(11.4km)', 'p222(15.0km)', 'vdcy(499.5km)']
Attempting BASE OBS from slac (attempt 1/3): https://geodesy.noaa.gov/corsdata/rinex/2021/118/slac/slac1180.21d.gz
✓ Success: Base OBS file written to slac1180.21d using slac
Attempting NAV file download from URL: https://igs.bkg.bund.de/root_ftp/IGS/BRDC/2021/118/BRDC00WRD_R_20211180000_01D_MN.rnx.gz
✓ NAV file downloaded
Processing: 2021-06-22-US-MTV-1
Station order for MTV: ['slac(11.4km)', 'p222(15.0km)', 'vdcy(499.5km)']
Attempting BASE OBS from slac (attempt 1/3): https://geodesy.noaa.gov/corsdata/rinex/2021/173/slac/slac1730.21d.gz
✓ Success: Base OBS file written to slac1730.21d using slac
Attempting NAV file download from URL: https://igs.bkg.bund.de/root_ftp/IGS/BRDC/2021/173/BRDC00WRD_R_20211730000_01D_MN.rnx.gz
✓ NAV file downloaded
Processing: 2021-08-12-US-MTV-1
Station order for MTV: ['slac(11.4km)', 'p222(15.0km)', 'vdcy(499.5km)']
Attempting BASE OBS from slac (atte

In [8]:
%%writefile C:\Users\gauth\Documents\SC4000_proj\ppk_phone_0510.conf
# ppk_phone_0510.conf - config file for RTKLIB PPK solution (RELAXED QUALITY CHECKS)
#2.9549
pos1-posmode       =kinematic  #
pos1-frequency     =l1+l2+l5   #
pos1-soltype       =combined-nophasereset #
pos1-elmask        =0          # (was 15) - Accept all satellite elevations
pos1-snrmask_r     =off        # (was on) - Don't filter by SNR on rover
pos1-snrmask_b     =off        # (was on) - Don't filter by SNR on base
pos1-snrmask_L1    =24,28,28,32,28,28,24,24,24
pos1-snrmask_L2    =34,34,34,34,34,34,34,34,34
pos1-snrmask_L5    =24,20,20,24,20,28,28,20,20
pos1-dynamics      =on         #
pos1-tidecorr      =on        #
pos1-ionoopt       =brdc       #
pos1-tropopt       =saas       #
pos1-sateph        =brdc       #
pos1-posopt1       =off        #
pos1-posopt2       =off        #
pos1-posopt3       =off        # 
pos1-posopt4       =off        # 
pos1-posopt5       =off        #
pos1-posopt6       =off        #
pos1-exclsats      =           # 
pos1-navsys        =13         # 
pos2-armode        =off        # 
pos2-gloarmode     =off        # 
pos2-bdsarmode     =on         # 
pos2-arfilter      =off        # (was on) - Don't filter AR solutions
pos2-arthres       =999        # (was 3) - Very relaxed AR threshold
pos2-arthresmin    =1.5
pos2-arthresmax    =10
pos2-arthres1      =0.25
pos2-arthres2      =0
pos2-arthres3      =1e-09
pos2-arthres4      =1e-05
pos2-varholdamb    =0.7        # (cyc^2)
pos2-gainholdamb   =0.01
pos2-arlockcnt     =0
pos2-minfixsats    =4
pos2-minholdsats   =5
pos2-mindropsats   =10
pos2-arelmask      =0             # (deg)
pos2-arminfix      =50
pos2-armaxiter     =1
pos2-elmaskhold    =0         # (was 15) - Accept all elevations for hold
pos2-aroutcnt      =4
pos2-maxage        =60         # (s) - INCREASED from 30 to 60 (already done)
pos2-syncsol       =off        #
pos2-slipthres     =0.1        #
pos2-dopthres      =10         # (INCREASED from 5 to 10 - RELAXED DOP CHECK)
pos2-rejionno      =1          # (m)
pos2-rejgdop       =30
pos2-niter         =1
pos2-baselen       =0          # (m)
pos2-basesig       =0          # (m)
out-solformat      =llh        # 
out-outhead        =on         #
out-outopt         =on         # 
out-outvel         =off        # 
out-timesys        =gpst       # 
out-timeform       =tow        # 
out-timendec       =3
out-degform        =deg        # 
out-fieldsep       =\n
out-outsingle      =off        # 
out-maxsolstd      =999        # (was 0) - Accept any solution quality
out-height         =ellipsoidal # 
out-geoid          =internal   # 
out-solstatic      =all        # 
out-nmeaintv1      =0          # (s)
out-nmeaintv2      =0          # (s)
out-outstat        =residual   # 
stats-eratio1      =1000       # (was 400) - Very relaxed error ratio L1
stats-eratio2      =1000       # (INCREASED from 300 to 1000 - RELAXED L2 RATIO)
stats-eratio5      =1000       # (INCREASED from 100 to 1000 - RELAXED L5 RATIO)
stats-errphase     =0.1        # (was 0.006) - Accept noisier phase
stats-errphaseel   =0.003      # (m)
stats-errphasebl   =0          # (m/10km)
stats-errdoppler   =1          # (Hz)
stats-snrmax       =50         # (dB.Hz)
stats-errsnr       =0          # (m)
stats-errrcv       =0          # ( )
stats-stdbias      =30         # (m)
stats-stdiono      =0.03       # (m)
stats-stdtrop      =0.3        # (m)
stats-prnaccelh    =1          # (m/s^2)
stats-prnaccelv    =0.1          # (m/s^2)
stats-prnbias      =0.032       # 
stats-prniono      =0.001      # (m)
stats-prntrop      =0.0001     # (m)
stats-prnpos       =0          # (m)
stats-clkstab      =5e-12      # (s/s)
ant1-postype       =llh        # 
ant1-pos1          =0          # 
ant1-pos2          =0          
ant1-pos3          =0          
ant1-anttype       =\n
ant1-antdele       =0          
ant1-antdeln       =0          
ant1-antdelu       =0          
ant2-postype       =posfile    
ant2-pos1          =0          
ant2-pos2          =0          
ant2-pos3          =0          
ant2-anttype       =\n
ant2-antdele       =0          
ant2-antdeln       =0          
ant2-antdelu       =0          
ant2-maxaveep      =1
ant2-initrst       =off         
misc-timeinterp    =off         
misc-sbasatsel     =0          
misc-rnxopt1       =\n
misc-rnxopt2       =\n
misc-pppopt        =\n
misc-svrcycle      =5         # (ms)
misc-timeout       =10000      # (ms)
misc-reconnect     =10000      # (ms)
misc-nmeacycle     =5000       # (ms)
misc-buffsize      =32768      # (bytes)
misc-navmsgsel     =all        # (0:all,1:rover,2:base,3:corr)
misc-proxyaddr     =\n
misc-fswapmargin   =30         # (s)
file-satantfile    =\n
file-rcvantfile    =\n
file-staposfile    =C:\\Users\\gauth\\Documents\\SC4000_proj\\bases.sta
file-geoidfile     =\n
file-ionofile      =\n
file-dcbfile       =\n
file-eopfile       =\n
file-blqfile       =\n
file-tempdir       =\n
file-geexefile     =\n
file-solstatfile   =\n
file-tracefile     =

Overwriting C:\Users\gauth\Documents\SC4000_proj\ppk_phone_0510.conf


In [15]:
"""
run_ppk_multi.py - convert raw android files to rinex and run PPK solutions for GDSC_2022
data set with RTKLIB and/or rtklib-py.   MODIFIED FOR TRAIN DATA
"""

import sys
import numpy as np
from os.path import join, isdir, isfile
from glob import glob
from multiprocessing import Pool
import subprocess
from time import time
import os, shutil

# --- PATH MODIFICATIONS TO MATCH YOUR ENVIRONMENT ---
# Append rtklib-py (for definition imports, though Py is disabled)
sys.path.append(r'C:\Users\gauth\Documents\SC4000_proj\rtklib-py')
# Append android_rinex source
sys.path.append(r'C:\Users\gauth\Documents\SC4000_proj\android_rinex\src')
# --------------------------------------------------

# Deferred imports that rely on sys.path being set correctly
try:
    import gnsslogger_to_rnx as rnx
except ImportError:
    print("FATAL ERROR: Could not import 'gnsslogger_to_rnx'. Please ensure the 'android_rinex\\src' folder is at the correct path specified above.")
    sys.exit(1)


# set run parameters
maxepoch = None # max number of epochs, used for debug, None = no limit

# Set solution choices
ENABLE_PY = False        # Use RTKLIB-PY to generate solutions 
ENABLE_RTKLIB = True     # Use RTKLIB to generate solutions
OVERWRITE_RINEX = True  # overwrite existing rinex files
OVERWRITE_SOL = True    # overwrite existing solution files
CLEAN_OLD_FILES = True  # Delete old generated files before regenerating

# --- DATA PATH ---
datadir = r'C:\Users\gauth\Documents\SC4000_proj\test' # Adjusted data directory
# The basefiles search must find the .20o files (RINEX v2 observation file for base)
# Example: slac1500.20o
basefiles = '../*0.2*o' 
# The navfiles search must find the BRDC file (RINEX v3 navigation file)
# Example: BRDC00WRD_R_20201500000_01D_MN.rnx
navfiles = '../*MN.rnx' 

# Setup for RTKLIB 
binpath_rtklib  = r'C:\Users\gauth\Documents\RTKLIB\rnx2rtkp.exe' # Adjusted executable path
cfgfile_rtklib = r'C:\Users\gauth\Documents\SC4000_proj\ppk_phone_0510.conf' # Adjusted config path
soltag_rtklib = '_rtklib' # postfix for solution file names

# Setup for rtklib-py (disabled)
cfgfile = r'D:\kaggle\GSDC2022\config\ppk_phone_0510.py' # This placeholder path doesn't matter since ENABLE_PY=False
soltag_py = '_py0510'  

PHONES = ['GooglePixel4', 'GooglePixel4XL', 'Pixel4Modded', 'GooglePixel5', 'GooglePixel6Pro', 'XiaomiMi8', 'SamsungGalaxyS20Ultra']

# Base stations we're actually using - WGS84 XYZ coordinates
BASE_POS = {
    'slac': [-2703115.9184, -4291767.2037, 3854247.9027],
    'vdcy': [-2497836.5139, -4654543.2609, 3563028.9379],
    'p222': [-2689640.2891, -4290437.3671, 3865050.9313],
}

# input structure for rinex conversion
class args:
    def __init__(self):
        # Input parameters for conversion to rinex
        self.slip_mask = 0 
        self.fix_bias = True
        self.timeadj = 1e-7
        self.pseudorange_bias = 0
        self.filter_mode = 'sync'
        # Optional hader values for rinex files
        self.marker_name = ''
        self.observer = ''
        self.agency = ''
        self.receiver_number = ''
        self.receiver_type = ''
        self.receiver_version = ''
        self.antenna_number = ''
        self.antenna_type = ''

# Copy and read config file (Disabled by ENABLE_PY=False)
if ENABLE_PY:
    import rinex as rn
    import rtkcmn as gn
    from rtkpos import rtkinit
    from postpos import procpos, savesol
    shutil.copyfile(cfgfile, '__ppk_config.py')
    import __ppk_config as cfg

# NEW FUNCTION: Delete old generated files
def clean_old_files(folder):
    """Delete old RINEX and solution files from folder"""
    os.chdir(folder)
    
    # Patterns for files to delete
    patterns_to_delete = [
        join('supplemental', '*.obs'),         # RINEX observation files
        join('supplemental', '*_rtklib.pos'),  # RTKLIB solution files
        join('supplemental', '*_py*.pos'),     # rtklib-py solution files
        join('supplemental', '*.trc'),         # RTKLIB trace files
        join('supplemental', '*.stat'),        # RTKLIB statistics files (if any)
    ]
    
    deleted_count = 0
    for pattern in patterns_to_delete:
        files = glob(pattern)
        for file in files:
            try:
                os.remove(file)
                deleted_count += 1
            except Exception as e:
                print(f"  Could not delete {file}: {e}")
    
    if deleted_count > 0:
        print(f"  Deleted {deleted_count} old file(s)")
    
    return deleted_count

# function to convert single rinex file
def convert_rnx(folder, rawFile, rovFile, slipMask):
    os.chdir(folder)
    argsIn = args()
    argsIn.input_log = rawFile
    argsIn.output = os.path.basename(rovFile)
    argsIn.slip_mask = slipMask
    rnx.convert2rnx(argsIn)

# function to run single RTKLIB-Py solution (disabled)
def run_ppk(folder, rovfile, basefile, navfile, solfile):
    return rovfile

# function to run single RTKLIB solution
def run_rtklib(folder, rovfile, basefile, navfile, solfile, progress_str=""):
    """Run RTKLIB PPK solution with progress tracking"""
    
    # Extract base station name from basefile
    base_station = os.path.basename(basefile).split('0')[0]
    
    # Get dataset/phone info from folder path
    path_parts = folder.split(os.sep)
    dataset_phone = "/".join(path_parts[-2:])  # e.g., "2020-05-15-US-MTV-1/GooglePixel4"
    
    print(f"\n{progress_str}")
    print(f"  Processing: {dataset_phone}")
    print(f"  Base Station: {base_station}")
    print(f"  Rover: {os.path.basename(rovfile)}")
    print(f"  Nav: {os.path.basename(navfile)}")
    
    # create command to run solution
    rtkcmd='%s -x 0 -y 2 -k %s -o %s %s %s %s' % \
        (binpath_rtklib, cfgfile_rtklib, solfile, rovfile, basefile, navfile)
    
    os.chdir(folder)
    
    try:
        result = subprocess.run(rtkcmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, 
                               check=True, timeout=120)
        
        # Check if .pos file was actually created
        if isfile(solfile):
            # Get file size for verification
            file_size = os.path.getsize(solfile)
            print(f"  ✓ SUCCESS: Generated {os.path.basename(solfile)} ({file_size} bytes)")
            return True
        else:
            print(f"  ✗ FAILED: .pos file not created")
            return False
            
    except subprocess.TimeoutExpired:
        print(f"  ✗ FAILED: Processing timeout (>120s)")
        return False
    except subprocess.CalledProcessError as e:
        print(f"  ✗ FAILED: RTKLIB error (exit code {e.returncode})")
        return False
    except Exception as e:
        print(f"  ✗ FAILED: {type(e).__name__}: {e}")
        return False

####### Start of main code ##########################

def main():

    # get list of data sets in data path
    datasets = np.sort(os.listdir(datadir))

    # loop through data set folders
    rinexIn = []
    ppkIn = []
    rtklibIn = []
    for dataset in datasets:
        for phone in PHONES:
            # skip if no folder for this phone
            folder = join(datadir, dataset, phone)
            if not isdir(folder):  
                continue
            
            # Change directory is important for glob() calls later
            try:
                os.chdir(folder)
            except FileNotFoundError:
                print(f"Directory not found: {folder}. Skipping.")
                continue

            # --- CLEAN OLD FILES IF REQUESTED ---
            if CLEAN_OLD_FILES:
                clean_old_files(folder)
            # ------------------------------------

            rawFile = join('supplemental', 'gnss_log.txt')
            rovFile = join('supplemental', 'gnss_log.obs')

            rinex = False
            # check if need rinex conversion
            if OVERWRITE_RINEX or not isfile(rovFile):
                # generate list of input parameters for each rinex conversion
                if phone == 'SamsungS20Ultra': 
                    slipMask = 0 
                else:
                    slipMask = 0 
                rinexIn.append((folder, rawFile, rovFile, slipMask))
                print(f"{dataset}/{phone}: Converting RINEX") 
                rinex = True
            
            # check if need to create PPK solution
            try:
                # glob looks in the parent directory (..) for the base/nav files
                baseFile = glob(basefiles)[0] 
                navFile = glob(navfiles)[0]   
                solFile = rovFile[:-4] + soltag_py + '.pos'
                solFile_rtklib = rovFile[:-4] + soltag_rtklib + '.pos'
            except IndexError:
                # This error means glob didn't find the necessary base or nav files in the parent directory
                print(f"{dataset}/{phone}: Missing base or nav file")
                continue
            except Exception as e:
                print(f'{dataset}/{phone}: Error - {e}')
                continue
            
            # Check if RTKLIB solution needs to run
            if ENABLE_RTKLIB and (OVERWRITE_SOL == True or 
                        len(glob(solFile_rtklib)) == 0 or rinex == True):
                # generate list of input/output files for each rtklib ppk solution
                print(f"{dataset}/{phone}: Queued for RTKLIB processing")
                rtklibIn.append((folder, rovFile, baseFile, navFile, solFile_rtklib))

    if len(rinexIn) > 0:
        print(f'\n{"="*60}')
        print(f'Converting {len(rinexIn)} RINEX files...')
        print(f'{"="*60}')
        # Run rinex conversion sequentially
        for idx, input in enumerate(rinexIn, 1):
            print(f"\n[{idx}/{len(rinexIn)}] Converting RINEX...")
            convert_rnx(input[0],input[1],input[2],input[3])
            print(f"  ✓ Conversion complete")

    if ENABLE_RTKLIB and len(rtklibIn) > 0:
        print(f'\n{"="*60}')
        print(f'Calculating {len(rtklibIn)} RTKLIB solutions...')
        print(f'{"="*60}')
        
        success_count = 0
        fail_count = 0
        
        # Run RTKLIB solutions sequentially
        for idx, input in enumerate(rtklibIn, 1):
            progress_str = f"[{idx}/{len(rtklibIn)}] Generating .pos file..."
            success = run_rtklib(input[0], input[1], input[2], input[3], input[4], progress_str)
            
            if success:
                success_count += 1
            else:
                fail_count += 1
        
        print(f'\n{"="*60}')
        print(f'RTKLIB Processing Summary:')
        print(f'  ✓ Successful: {success_count}')
        print(f'  ✗ Failed: {fail_count}')
        print(f'  Total: {len(rtklibIn)}')
        print(f'{"="*60}')

    print(f'\n{"="*60}')
    print('PROCESSING COMPLETE')
    print(f'{"="*60}')

if __name__ == '__main__':
    t0 = time()
    main()
    print(f'\nTotal Runtime: {time() - t0:.1f} seconds')

  Deleted 4 old file(s)
2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra: Converting RINEX
2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra: Queued for RTKLIB processing
  Deleted 4 old file(s)
2021-06-22-US-MTV-1/XiaomiMi8: Converting RINEX
2021-06-22-US-MTV-1/XiaomiMi8: Queued for RTKLIB processing
  Deleted 4 old file(s)
2021-08-12-US-MTV-1/GooglePixel4: Converting RINEX
2021-08-12-US-MTV-1/GooglePixel4: Queued for RTKLIB processing
  Deleted 4 old file(s)
2021-08-17-US-MTV-1/GooglePixel5: Converting RINEX
2021-08-17-US-MTV-1/GooglePixel5: Queued for RTKLIB processing
  Deleted 4 old file(s)
2021-08-24-US-SVL-2/GooglePixel5: Converting RINEX
2021-08-24-US-SVL-2/GooglePixel5: Queued for RTKLIB processing
  Deleted 4 old file(s)
2021-09-07-US-MTV-1/SamsungGalaxyS20Ultra: Converting RINEX
2021-09-07-US-MTV-1/SamsungGalaxyS20Ultra: Queued for RTKLIB processing
  Deleted 4 old file(s)
2021-09-14-US-MTV-1/GooglePixel5: Converting RINEX
2021-09-14-US-MTV-1/GooglePixel5: Queued for RTKLIB processing
  De

In [16]:
"""
check_pos_file_status.py - Check all .pos files to see if they have data or just headers
"""

import os
from os.path import join, isfile, isdir

datadir = r'C:\Users\gauth\Documents\SC4000_proj\test'
pos_suffix = '_rtklib.pos'

PHONES = ['GooglePixel4', 'GooglePixel4XL', 'Pixel4Modded', 'GooglePixel5', 
          'GooglePixel6Pro', 'XiaomiMi8', 'SamsungGalaxyS20Ultra']

def check_pos_file(filepath):
    """
    Check if .pos file has data or is just a header
    Returns: (status, file_size, data_line_count)
    status: 'GOOD', 'HEADER_ONLY', 'MISSING', 'SMALL'
    """
    if not isfile(filepath):
        return 'MISSING', 0, 0
    
    file_size = os.path.getsize(filepath)
    
    # Count data lines (non-header, non-empty)
    try:
        with open(filepath, 'r', errors='ignore') as f:
            lines = f.readlines()
            data_lines = [l for l in lines if not l.startswith('%') and l.strip()]
            data_count = len(data_lines)
    except:
        return 'ERROR', file_size, 0
    
    # Classification
    if file_size <= 1020:  # Approximately header size
        return 'HEADER_ONLY', file_size, data_count
    elif data_count == 0:
        return 'HEADER_ONLY', file_size, 0
    elif data_count < 100:
        return 'SMALL', file_size, data_count
    else:
        return 'GOOD', file_size, data_count

def main():
    os.chdir(datadir)
    datasets = sorted(os.listdir())
    
    # Statistics
    good_count = 0
    header_only_count = 0
    small_count = 0
    missing_count = 0
    error_count = 0
    
    header_only_files = []
    small_files = []
    missing_files = []
    
    print("="*70)
    print("CHECKING ALL .POS FILES")
    print("="*70)
    print(f"{'Dataset/Phone':<45} {'Status':<15} {'Size':>10} {'Lines':>8}")
    print("-"*70)
    
    for dataset in datasets:
        if not isdir(dataset):
            continue
        
        for phone in PHONES:
            folder = join(datadir, dataset, phone)
            
            if not isdir(folder):
                continue
            
            # Check for .pos file
            pos_file = join(folder, 'supplemental', f'gnss_log{pos_suffix}')
            
            status, size, lines = check_pos_file(pos_file)
            
            # Format the path for display
            path_display = f"{dataset}/{phone}"
            
            # Color coding with symbols
            if status == 'GOOD':
                symbol = "✓"
                good_count += 1
                # Only print if verbose mode, otherwise skip good files
                # print(f"{path_display:<45} {symbol} {status:<13} {size:>10} {lines:>8}")
            elif status == 'HEADER_ONLY':
                symbol = "✗"
                print(f"{path_display:<45} {symbol} {status:<13} {size:>10} {lines:>8}")
                header_only_count += 1
                header_only_files.append(path_display)
            elif status == 'SMALL':
                symbol = "⚠"
                print(f"{path_display:<45} {symbol} {status:<13} {size:>10} {lines:>8}")
                small_count += 1
                small_files.append(path_display)
            elif status == 'MISSING':
                symbol = "✗"
                print(f"{path_display:<45} {symbol} {status:<13} {size:>10} {lines:>8}")
                missing_count += 1
                missing_files.append(path_display)
            else:  # ERROR
                symbol = "!"
                print(f"{path_display:<45} {symbol} {status:<13} {size:>10} {lines:>8}")
                error_count += 1
    
    # Summary
    print("="*70)
    print("SUMMARY")
    print("="*70)
    total = good_count + header_only_count + small_count + missing_count + error_count
    print(f"Total files checked:    {total}")
    print(f"  ✓ Good (>100 lines):  {good_count}")
    print(f"  ⚠ Small (<100 lines): {small_count}")
    print(f"  ✗ Header only:        {header_only_count}")
    print(f"  ✗ Missing:            {missing_count}")
    if error_count > 0:
        print(f"  ! Errors:             {error_count}")
    
    # Detailed problem files
    if header_only_files:
        print("\n" + "="*70)
        print(f"HEADER-ONLY FILES ({len(header_only_files)}):")
        print("="*70)
        for f in header_only_files:
            print(f"  • {f}")
    
    if small_files:
        print("\n" + "="*70)
        print(f"SMALL FILES - May need investigation ({len(small_files)}):")
        print("="*70)
        for f in small_files:
            print(f"  • {f}")
    
    if missing_files:
        print("\n" + "="*70)
        print(f"MISSING FILES ({len(missing_files)}):")
        print("="*70)
        for f in missing_files:
            print(f"  • {f}")
    
    # Success rate
    if total > 0:
        success_rate = (good_count / total) * 100
        print("\n" + "="*70)
        print(f"SUCCESS RATE: {success_rate:.1f}% ({good_count}/{total} files)")
        print("="*70)

if __name__ == '__main__':
    main()

CHECKING ALL .POS FILES
Dataset/Phone                                 Status                Size    Lines
----------------------------------------------------------------------
SUMMARY
Total files checked:    36
  ✓ Good (>100 lines):  36
  ⚠ Small (<100 lines): 0
  ✗ Header only:        0
  ✗ Missing:            0

SUCCESS RATE: 100.0% (36/36 files)


In [4]:
"""
check_time_overlap.py - Check if rover and base station times overlap
"""

import os
from os.path import join
import numpy as np

datadir = r'C:\Users\gauth\Documents\SC4000_proj\train'

failed_datasets = [
    '2020-06-24-US-MTV-1',
    '2020-08-03-US-MTV-2',
    '2020-11-23-US-MTV-1',
]

def parse_rinex_time_range(filepath):
    """Parse RINEX file to get time range"""
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()
        
        # Find END OF HEADER
        header_end = 0
        for i, line in enumerate(lines):
            if 'END OF HEADER' in line:
                header_end = i + 1
                break
        
        if header_end == 0:
            return None, None
        
        # Get first and last observation epochs
        first_time = None
        last_time = None
        
        for line in lines[header_end:]:
            # RINEX observation epoch line starts with year
            if len(line) > 26 and line[0] == ' ' and line[1:3].strip().isdigit():
                # Parse time: YY MM DD HH MM SS
                try:
                    year = int('20' + line[1:3].strip())
                    month = int(line[4:6])
                    day = int(line[7:9])
                    hour = int(line[10:12])
                    minute = int(line[13:15])
                    second = float(line[16:26])
                    
                    time_str = f"{year}-{month:02d}-{day:02d} {hour:02d}:{minute:02d}:{second:06.3f}"
                    
                    if first_time is None:
                        first_time = time_str
                    last_time = time_str
                except:
                    continue
        
        return first_time, last_time
    except Exception as e:
        print(f"    Error reading file: {e}")
        return None, None

for dataset in failed_datasets:
    print(f"\n{'='*60}")
    print(f"Checking: {dataset}")
    print(f"{'='*60}")
    
    # Check base station file
    os.chdir(join(datadir, dataset))
    base_files = [f for f in os.listdir() if f.endswith('.20o')]
    
    if base_files:
        base_file = base_files[0]
        print(f"\n📡 Base Station: {base_file}")
        first, last = parse_rinex_time_range(base_file)
        if first and last:
            print(f"    First epoch: {first}")
            print(f"    Last epoch:  {last}")
        else:
            print(f"    ⚠️  Could not parse time range")
    
    # Check rover files for one phone
    phone = 'GooglePixel4'
    rover_file = join(datadir, dataset, phone, 'supplemental', 'gnss_log.obs')
    
    if os.path.exists(rover_file):
        print(f"\n📱 Rover: {phone}")
        first, last = parse_rinex_time_range(rover_file)
        if first and last:
            print(f"    First epoch: {first}")
            print(f"    Last epoch:  {last}")
        else:
            print(f"    ⚠️  Could not parse time range")

print("\n" + "="*60)
print("TIME RANGE CHECK COMPLETE")
print("="*60)


Checking: 2020-06-24-US-MTV-1

📡 Base Station: slac1760.20o
    First epoch: 2020-06-24 00:00:00.000
    Last epoch:  2020-06-24 23:59:30.000

📱 Rover: GooglePixel4
    ⚠️  Could not parse time range

Checking: 2020-08-03-US-MTV-2

📡 Base Station: slac2160.20o
    First epoch: 2020-08-03 00:00:00.000
    Last epoch:  2020-08-03 23:59:30.000

📱 Rover: GooglePixel4
    ⚠️  Could not parse time range

Checking: 2020-11-23-US-MTV-1

📡 Base Station: slac3280.20o
    First epoch: 2020-11-23 00:00:00.000
    Last epoch:  2020-11-23 23:59:30.000

TIME RANGE CHECK COMPLETE


In [20]:
import os
from os.path import join, isfile, basename
import pandas as pd

# -------------------------- UPDATE THESE TWO PATHS --------------------------

# 1. The directory containing your final, flat .pos.txt files
POS_ROOT_DIR = r'C:\Users\gauth\Documents\SC4000_proj\pos_output_test'

# 2. The root directory containing the original nested RINEX data
RINEX_ROOT_DIR = r'C:\Users\gauth\Documents\SC4000_proj\test'

# -----------------------------------------------------------------------------

def count_rinex_epochs(filepath):
    """Counts data lines in a RINEX observation file (non-header lines) using V3/V4 standard"""
    count = 0
    try:
        with open(filepath, 'r', errors='ignore') as f:
            for line in f:
                # FIX: Check for the RINEX V3/V4 epoch block start marker: '>'
                if line.startswith('>'):
                    count += 1
    except:
        return 0
    return count

def count_pos_epochs(filepath):
    """Counts data lines in an RTKLIB .pos file (non-header lines)"""
    count = 0
    try:
        with open(filepath, 'r', errors='ignore') as f:
            for line in f:
                # RTKLIB solution lines do NOT start with '%' or are empty
                if not line.startswith('%') and line.strip():
                    count += 1
    except:
        return 0
    return count

def find_missing_epochs():
    results = []
    
    print("🔎 Comparing expected RINEX epochs to actual RTKLIB solutions...")
    print(f"Checking POS directory: {POS_ROOT_DIR}")
    print(f"Checking RINEX root: {RINEX_ROOT_DIR}\n")
    
    # Iterate over all .pos.txt files in the flat directory
    for filename in os.listdir(POS_ROOT_DIR):
        if not (filename.endswith('-pos.txt') or filename.endswith('_pos.txt')):
            continue

        pos_file_path = join(POS_ROOT_DIR, filename)
        
        # --- PATH PARSING (This part is now correct) ---
        parts = filename.replace('-pos.txt', '').split('-')
        
        if len(parts) >= 6:
            dataset_name = '-'.join(parts[:6])
            phone_name = parts[-1] 
        else:
            print(f"🚨 SKIP: Could not parse dataset/phone name from {filename}")
            continue
        
        rov_path = join(RINEX_ROOT_DIR, dataset_name, phone_name, 'supplemental', 'gnss_log.obs')
        
        if not isfile(rov_path):
            print(f"🚨 FILE NOT FOUND: Expected RINEX at {rov_path}")
            continue
        
        # --- END PATH PARSING ---

        expected_epochs = count_rinex_epochs(rov_path)
        actual_epochs = count_pos_epochs(pos_file_path)
        deficit = expected_epochs - actual_epochs
        
        results.append({
            'Dataset/Phone': f'{dataset_name}/{phone_name}',
            'Expected': expected_epochs,
            'Actual': actual_epochs,
            'Deficit': deficit
        })
        print(f"Processed: {dataset_name}/{phone_name} | Expected: {expected_epochs:,} | Actual: {actual_epochs:,} | Deficit: {deficit:,}")


    if not results:
        print("\n=========================================================================")
        print("❌ ERROR: RESULTS LIST IS EMPTY. Check that both root paths are correct.")
        print("=========================================================================")
        return 

    # Display Results
    df = pd.DataFrame(results)
    total_expected = df['Expected'].sum()
    total_actual = df['Actual'].sum()
    total_deficit = total_expected - total_actual
    
    # The actual goal of finding the 380 deficit
    if total_deficit < 0:
        total_deficit = 0 # Ignore negative deficits for the goal check

    print("\n" + "="*80)
    print(f"FINAL CHECK: Expected Total={total_expected:,}, Actual Total={total_actual:,}, Deficit={total_deficit:,}")
    print("="*80)
    
    df_deficit = df[df['Deficit'] > 0].sort_values(by='Deficit', ascending=False)
    
    if total_deficit > 0:
        print("\n🚨 Files with Epoch Deficits (Sorted by Missing Count):")
        print(df_deficit.to_string(index=False))
        
        culprit = df_deficit.iloc[0]['Dataset/Phone']
        print(f"\n💡 Problematic Dataset Identified: {culprit}")
        print(f"   Missing {df_deficit.iloc[0]['Deficit']} epochs.")
        return culprit
    else:
        print("\n🎉 All expected epochs accounted for (or actual solutions exceed expected).")
    
    return None

if __name__ == '__main__':
    try:
        import pandas as pd
        find_missing_epochs()
    except ImportError:
        print("This script requires the 'pandas' library. Please install it with: pip install pandas")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

🔎 Comparing expected RINEX epochs to actual RTKLIB solutions...
Checking POS directory: C:\Users\gauth\Documents\SC4000_proj\pos_output_test
Checking RINEX root: C:\Users\gauth\Documents\SC4000_proj\test

Processed: 2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra | Expected: 1,724 | Actual: 1,724 | Deficit: 0
Processed: 2021-06-22-US-MTV-1/XiaomiMi8 | Expected: 1,398 | Actual: 1,397 | Deficit: 1
Processed: 2021-08-12-US-MTV-1/GooglePixel4 | Expected: 1,265 | Actual: 1,250 | Deficit: 15
Processed: 2021-08-17-US-MTV-1/GooglePixel5 | Expected: 1,673 | Actual: 1,673 | Deficit: 0
Processed: 2021-08-24-US-SVL-2/GooglePixel5 | Expected: 3,314 | Actual: 3,308 | Deficit: 6
Processed: 2021-09-07-US-MTV-1/SamsungGalaxyS20Ultra | Expected: 1,802 | Actual: 1,800 | Deficit: 2
Processed: 2021-09-14-US-MTV-1/GooglePixel5 | Expected: 1,270 | Actual: 1,270 | Deficit: 0
Processed: 2021-09-20-US-MTV-1/XiaomiMi8 | Expected: 1,795 | Actual: 1,795 | Deficit: 0
Processed: 2021-09-20-US-MTV-2/GooglePixel4 | Expected